# Praktikum Pengolahan Citra · Minggu 2
## Sub-CPMK 1: Citra sebagai Data

**Tujuan sesi ini:** setelah praktikum ini, kamu bisa menjelaskan citra digital sebagai matriks angka, membedakan piksel, *channel*, resolusi spasial, dan resolusi intensitas, serta memprediksi efek sampling dan kuantisasi terhadap kualitas citra.

**Cara pakai notebook ini:** jalankan tiap sel dari atas ke bawah (`Shift+Enter`). Sel bertanda **🔧 Latihan** berisi kode yang belum lengkap, bagian `# TODO` harus kamu isi sendiri. Jangan lompat ke sel berikutnya sebelum sel sebelumnya selesai dijalankan.

---

## Bagian 0 · Persiapan

Jalankan dua sel di bawah ini sekali di awal sesi.

In [ ]:
# Kalau ada modul yang belum terpasang (biasanya tidak perlu di Google Colab), uncomment baris berikut:
# !pip install -q opencv-python numpy matplotlib scikit-image

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data as skdata

%matplotlib inline

print("OpenCV :", cv2.__version__)
print("NumPy  :", np.__version__)

Citra contoh diambil dari `scikit-image` (bawaan, tidak perlu unduh manual) lalu disimpan sebagai file `citra.jpg`, supaya kita bisa memuatnya dengan `cv2.imread` seperti citra pada umumnya.

Kalau kamu mau pakai foto sendiri, jalankan sel **opsional** setelahnya (unggah lewat kotak dialog). Semua kode setelah itu otomatis memakai citra yang baru diunggah, karena semuanya memuat ulang dari `citra.jpg`.

In [ ]:
# Simpan citra contoh dari scikit-image sebagai file, supaya bisa dimuat dengan cv2.imread
rgb_sample = skdata.astronaut()  # array RGB, uint8, 512x512x3
cv2.imwrite("citra.jpg", cv2.cvtColor(rgb_sample, cv2.COLOR_RGB2BGR))
print("citra.jpg tersimpan, ukuran:", rgb_sample.shape)

In [ ]:
# OPSIONAL: jalankan sel ini kalau mau pakai foto sendiri (menimpa citra.jpg)
from google.colab import files
import shutil

uploaded = files.upload()
nama_file = list(uploaded.keys())[0]
shutil.copy(nama_file, "citra.jpg")
print("Terpasang sebagai citra.jpg")

## Bagian 1 · Anatomi Citra Digital

Bagi OpenCV, citra hanyalah array angka tiga dimensi: (tinggi, lebar, jumlah *channel*). Mari lihat langsung.

In [ ]:
img = cv2.imread("citra.jpg")

print("Tipe data        :", type(img))
print("Shape             :", img.shape)   # (tinggi, lebar, channel)
print("dtype             :", img.dtype)   # uint8 -> nilai 0-255
print("Ukuran raw        :", round(img.nbytes / 1024, 1), "KB (belum dikompresi)")
print("Ukuran file di disk:", round(os.path.getsize("citra.jpg") / 1024, 1), "KB (setelah kompresi JPEG)")

### ⚠️ Jebakan klasik: BGR, bukan RGB

`cv2.imread` membaca citra dalam urutan **B-G-R**, bukan R-G-B seperti kebanyakan library lain (termasuk matplotlib). Kalau kita langsung `plt.imshow(img)` tanpa konversi, warnanya akan salah. Coba lihat sendiri:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].imshow(img)  # salah! ini masih BGR
ax[0].set_title("plt.imshow(img) -> BGR ditampilkan\napa adanya (SALAH)")
ax[0].axis("off")

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
ax[1].imshow(img_rgb)
ax[1].set_title("setelah cv2.cvtColor(...BGR2RGB)\n(BENAR)")
ax[1].axis("off")

plt.tight_layout()
plt.show()

Perhatikan warna kulit di gambar kiri jadi pucat kebiruan, tidak seperti warna kulit natural di gambar kanan. Mulai sekarang kita pakai `img_rgb` untuk semua keperluan tampilan dengan matplotlib. Tapi ingat, fungsi OpenCV lain (misalnya `cv2.imwrite`, `cv2.rectangle`) tetap mengharapkan urutan BGR.

## Bagian 2 · Piksel sebagai Angka

Satu piksel cuma tiga angka untuk citra RGB: intensitas merah, hijau, biru, masing-masing 0–255. Mari akses dan ubah piksel secara langsung lewat *indexing* NumPy.

In [ ]:
baris, kolom = 100, 150
print("Nilai piksel di (baris=100, kolom=150):", img_rgb[baris, kolom])
print("  -> R =", img_rgb[baris, kolom, 0])
print("  -> G =", img_rgb[baris, kolom, 1])
print("  -> B =", img_rgb[baris, kolom, 2])

In [ ]:
img_modif = img_rgb.copy()

# Hitamkan blok 60x60 piksel di tengah citra
h, w, _ = img_modif.shape
pusat_y, pusat_x = h // 2, w // 2
img_modif[pusat_y-30:pusat_y+30, pusat_x-30:pusat_x+30] = [0, 0, 0]

plt.figure(figsize=(4, 4))
plt.imshow(img_modif)
plt.title("Blok 60x60 piksel dihitamkan manual")
plt.axis("off")
plt.show()

### 🔧 Latihan 1

Ganti blok hitam di atas menjadi **kotak putih** berukuran 40x40 piksel di **pojok kiri atas** citra (bukan di tengah). Lengkapi `# TODO` di bawah ini.

In [ ]:
img_latihan1 = img_rgb.copy()

# TODO: ubah img_latihan1[0:40, 0:40] menjadi putih, yaitu [255, 255, 255]


plt.figure(figsize=(4, 4))
plt.imshow(img_latihan1)
plt.title("Latihan 1: kotak putih di pojok kiri atas")
plt.axis("off")
plt.show()

## Bagian 3 · Membedah Channel RGB

Tiga *channel* bisa dipisah dan dilihat satu per satu sebagai citra abu-abu.

In [ ]:
R, G, B = cv2.split(img_rgb)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(img_rgb);        ax[0].set_title("Asli (RGB)");     ax[0].axis("off")
ax[1].imshow(R, cmap="gray"); ax[1].set_title("Channel R saja"); ax[1].axis("off")
ax[2].imshow(G, cmap="gray"); ax[2].set_title("Channel G saja"); ax[2].axis("off")
ax[3].imshow(B, cmap="gray"); ax[3].set_title("Channel B saja"); ax[3].axis("off")
plt.tight_layout()
plt.show()

Perhatikan: objek yang warnanya dominan merah akan tampak **paling terang** pada citra channel-R, dan biasanya paling gelap pada citra channel-B. Ini prinsip yang nanti kita pakai lagi untuk deteksi objek berbasis warna di Minggu 4.

### 🔧 Latihan 2

Gabungkan ulang ketiga *channel* di atas memakai `cv2.merge`, tapi **tukar posisi channel R dan B**. Tampilkan hasilnya. Menurutmu, apa yang terjadi pada warna citra, dan kenapa?

In [ ]:
# TODO: gabungkan ulang dengan urutan channel B, G, R (bukan R, G, B) memakai cv2.merge
tertukar = None  # ganti None dengan hasil cv2.merge(...)

plt.figure(figsize=(4, 4))
plt.imshow(tertukar)
plt.title("Latihan 2: channel R dan B ditukar")
plt.axis("off")
plt.show()

## Bagian 4 · Sampling: Resolusi Spasial

Sampling menentukan berapa banyak titik piksel yang mewakili citra. Mari perkecil citra secara drastis (memakai *nearest neighbor*, supaya benar-benar cuma "mengambil sampel", tanpa pembauran nilai antar piksel), lalu besarkan lagi ke ukuran semula, dan amati apa yang hilang.

In [ ]:
h, w = img_rgb.shape[:2]
skala_list = [1.0, 0.25, 0.10, 0.05]

fig, ax = plt.subplots(1, len(skala_list), figsize=(16, 4))
for i, skala in enumerate(skala_list):
    kecil = cv2.resize(img_rgb, (int(w*skala), int(h*skala)), interpolation=cv2.INTER_NEAREST)
    balik = cv2.resize(kecil, (w, h), interpolation=cv2.INTER_NEAREST)
    ax[i].imshow(balik)
    ax[i].set_title(f"skala {skala:.0%}\n({kecil.shape[1]}x{kecil.shape[0]} piksel)")
    ax[i].axis("off")
plt.tight_layout()
plt.show()

Di skala 5%, citra hanya disusun dari sekitar 25x25 piksel sebelum dibesarkan kembali. Detail wajah dan tekstur sudah hilang total, hanya blok-blok warna kasar yang tersisa, karena piksel sesedikit itu tidak cukup untuk merekam detail aslinya. Itulah konsekuensi langsung dari sampling yang terlalu jarang.

## Bagian 5 · Kuantisasi: Resolusi Intensitas

Kuantisasi menentukan berapa banyak **level kecerahan** yang dipakai, terpisah dari resolusi spasial. Supaya efeknya kelihatan jelas, kita pakai citra gradasi buatan sendiri (bukan foto), karena gradasi halus paling mudah menampakkan *false contouring*.

In [ ]:
# Citra gradasi horizontal 0 -> 255, sengaja dibuat halus
gradasi = np.tile(np.linspace(0, 255, 256, dtype=np.uint8), (100, 1))

def quantize(citra, levels):
    # Bulatkan tiap nilai piksel ke salah satu dari `levels` level intensitas
    step = 256 // levels
    return (citra // step * step).astype(np.uint8)

level_list = [256, 16, 8, 4, 2]
fig, ax = plt.subplots(len(level_list), 1, figsize=(8, 8))
for i, lv in enumerate(level_list):
    hasil = gradasi if lv == 256 else quantize(gradasi, lv)
    ax[i].imshow(hasil, cmap="gray", vmin=0, vmax=255, aspect="auto")
    ax[i].set_title(f"{lv} level intensitas", loc="left", fontsize=10)
    ax[i].axis("off")
plt.tight_layout()
plt.show()

Pada 4 dan 2 level, gradasi halus berubah jadi pita-pita tegas: itulah *false contouring*. Mata manusia bisa membedakan sekitar 100 level abu-abu dalam kondisi normal, jauh di bawah 256 level yang disediakan citra 8-bit, tapi di bawah sekitar 16 level pita-pitanya mulai terlihat jelas oleh mata telanjang.

### 🔧 Latihan 3

Terapkan fungsi `quantize` ke citra foto asli (`img_rgb`, bukan gradasi buatan), dengan 8 level dan 2 level. Bandingkan berdampingan dengan citra aslinya. Apakah *false contouring*-nya sejelas pada citra gradasi? Menurutmu kenapa bisa berbeda?

In [ ]:
# TODO: terapkan quantize() ke img_rgb dengan 8 level, simpan ke variabel `foto_8level`
foto_8level = None

# TODO: terapkan quantize() ke img_rgb dengan 2 level, simpan ke variabel `foto_2level`
foto_2level = None

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img_rgb);      ax[0].set_title("Asli (256 level)"); ax[0].axis("off")
ax[1].imshow(foto_8level);  ax[1].set_title("8 level");          ax[1].axis("off")
ax[2].imshow(foto_2level);  ax[2].set_title("2 level");          ax[2].axis("off")
plt.tight_layout()
plt.show()

## Bagian 6 · Tugas: Image Inspector

Gabungkan semua yang sudah dipelajari jadi satu fungsi `image_inspector(path)` yang:

1. Memuat citra dari `path` memakai `cv2.imread`
2. Mencetak dimensi, jumlah *channel*, dtype, dan ukuran file dalam KB
3. Menampilkan citra asli berdampingan dengan hasil kuantisasi 8 level dan 2 level

Kumpulkan file `.ipynb` ini (yang sudah kamu jalankan semua selnya, termasuk Latihan 1–3 dan tugas ini) sesuai instruksi pengumpulan dari asisten.

In [ ]:
def image_inspector(path):
    # TODO: lengkapi fungsi ini sesuai instruksi di atas
    citra = cv2.imread(path)
    citra_rgb = cv2.cvtColor(citra, cv2.COLOR_BGR2RGB)

    print("Dimensi     :", citra.shape)
    print("Channel     :", citra.shape[2] if citra.ndim == 3 else 1)
    print("dtype       :", citra.dtype)
    print("Ukuran file :", round(citra.nbytes / 1024, 1), "KB")

    # TODO: tampilkan citra asli, hasil kuantisasi 8 level, dan 2 level berdampingan


image_inspector("citra.jpg")

## Ringkasan & Minggu Depan

Hari ini kita membedah citra digital jadi angka: piksel, *channel*, urutan BGR OpenCV, resolusi spasial (sampling), dan resolusi intensitas (kuantisasi). Minggu depan (Sub-CPMK 2) kita pakai histogram untuk membaca kecerahan dan kontras citra tanpa perlu melihat gambarnya langsung.